> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 02 · AWS essentials: accounts, IAM, regions, CLI

**CampusMate v0**  ·  ⏱️ Time budget: **10 min theory + 20 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

CampusMate will run on Amazon Web Services (AWS). To deploy anything you must know *which account* you are in, *who* you are, *where* (region) things run, and how to read a 'permission denied' message without panicking. This is the exam-cell desk of the cloud: get your hall ticket checked before you sit the exam.

### What you'll do
- Confirm your identity with `aws sts get-caller-identity`
- Set and check your region
- Trigger one deliberate permission error and fix it
- Run the one-command setup check

## Settings

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib

REGION = "us-east-1"                                 # the AWS region we use in this workshop
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite (CampusMate's brain)
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # same model for quick experiments

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"                             # the fictional campus data
print("Region:", REGION, "| Model:", MODEL_ID)

## Accounts, IAM and regions in 4 lines

- **AWS account**: a container for everything you create and the bill for it. Your class shares one sandbox account.
- **IAM (Identity and Access Management)**: *who* can do *what* on *which* resource. Your login is an **IAM user**; services use **IAM roles**.
- **Policy**: a JSON document listing allowed actions, e.g. `bedrock:InvokeModel`. No allow = denied.
- **Region**: a physical AWS location like `us-east-1` (N. Virginia) or `ap-south-1` (Mumbai). Not every service is in every region.

🏫 **Analogy:** the account is the college; IAM is the ID-card system; a policy is the list of rooms your card opens;
a region is which campus you're on.

## Why we use us-east-1

Amazon Bedrock **AgentCore** (Runtime, Memory, Gateway, Policy, Evaluations) was verified in **us-east-1** for this workshop.
The Nova inference profile we use starts with `us.` because they belong to US regions. If you move to Mumbai (`ap-south-1`) you'd use
`apac.` profiles instead, but check AgentCore availability first.

## The AWS CLI

The **CLI (Command Line Interface)** lets you talk to AWS from a terminal. `aws configure` stores your access key in
`~/.aws/credentials` (outside this repo, never in `.env`, never in git).

## 🔬 Live check: who am I?

The `!` runs a terminal command from the notebook.

In [ ]:
!aws sts get-caller-identity
!aws configure get region

In [ ]:
import boto3
me = boto3.client("sts").get_caller_identity()
print("Account :", me["Account"])
print("I am    :", me["Arn"])
print("Region  :", REGION, "(set in the Settings cell)")

## A deliberate permission error

We create a temporary 'visitor pass' (a federation token) that is **only** allowed to call `sts:GetCallerIdentity`,
then try to list Bedrock models with it. AWS will say no. Reading that message is a core skill.

*(This needs IAM-user access keys. If you log in with SSO, the cell explains and skips.)*

In [ ]:
import json
from botocore.exceptions import ClientError

def visitor_session(policy):
    c = boto3.client("sts").get_federation_token(Name="campusmate-demo", Policy=json.dumps(policy), DurationSeconds=900)["Credentials"]
    return boto3.Session(aws_access_key_id=c["AccessKeyId"], aws_secret_access_key=c["SecretAccessKey"],
                         aws_session_token=c["SessionToken"], region_name=REGION)

only_whoami = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": "sts:GetCallerIdentity", "Resource": "*"}]}
try:
    visitor = visitor_session(only_whoami)
    visitor.client("bedrock").list_foundation_models()
except ClientError as e:
    print("ERROR CODE:", e.response["Error"]["Code"])
    print("MESSAGE   :", e.response["Error"]["Message"][:300])

**How to read it:** *User* `...federated-user/campusmate-demo` *is not authorized to perform* `bedrock:ListFoundationModels`.
That tells you **who** and **which action**. The fix is always: add that action to the identity's policy (or ask your admin).

## ✍️ Your turn

Fix the error: add the missing action to the visitor's policy so the call succeeds.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
fixed_policy = {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": ["sts:GetCallerIdentity"], "Resource": "*"},   # TODO: add "bedrock:ListFoundationModels" to this list
]}
try:
    models = visitor_session(fixed_policy).client("bedrock").list_foundation_models()["modelSummaries"]
    print(f"Fixed! The visitor can now see {len(models)} models.")
except ClientError as e:
    print("Still denied (expected until you fix the TODO):", e.response["Error"]["Code"])

## 🔬 Run the full setup check

This is the same script you ran during pre-work. Every line should say PASS.

In [ ]:
!{sys.executable} ../scripts/check_setup.py

## ✅ Checkpoint

You are on track if:
- `get-caller-identity` shows the sandbox account and your IAM user
- The visitor call failed with `AccessDeniedException`, then worked after your fix
- `check_setup.py` ends with 'All checks passed'

**Stuck or behind?** Copy from `scripts/02_aws_identity_check.py and scripts/check_setup.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `Unable to locate credentials` | `aws configure` not done | Run `aws configure`, paste your keys, region `us-east-1` |
| `ExpiredToken` / `InvalidClientTokenId` | Keys expired or mistyped | Ask for fresh keys and run `aws configure` again |
| `AccessDenied ... sts:GetFederationToken` | Your user can't create visitor passes | Skip that demo; read the error format from the slide instead |
| `'aws' is not recognized` | AWS CLI v2 not installed or terminal not restarted | Install AWS CLI v2 (SETUP.md) and reopen the terminal |